# 03 | What is known at each cutoff?

*BDCC project, Group G. Run `01_land_raw` first.*

Finance closes payment day D twice: at **C1** (08:00 UTC on D+1) and at **C2** (08:00 UTC on D+2). A later delivery can change the answer, so each close may only see what had arrived and was effective by its cutoff.

For each cutoff and each source this notebook gives every delivered row exactly one status:

| Status | Meaning | Goes into the close? |
|---|---|---|
| `REJECTED` | failed a row check in notebook 01 | no, kept with its reason |
| `NOT_YET_EFFECTIVE` | delivered by the cutoff, effective only after it | no, not yet |
| `REPLAY` | same key, version and content as a row already kept | no, the kept row stands for it |
| `SUPERSEDED` | a higher version of the same fact is available | no, replaced |
| `QUARANTINED` | the highest version has two different contents | no, unresolved conflict |
| `CURRENT` | the fact as known at the cutoff | **yes** |

Rows delivered after the cutoff are not in that snapshot at all.

**Where this fits in the brief**

- **Configuration B:** produce the close as of C1 and C2 and preserve both snapshots. This notebook builds the inputs of both closes. The close itself (reconciliation, exception queue, daily close) reads the `clean_*` views.
- **Close policy implemented here:** delivered at or before the cutoff and effective by then; a replay contributes once; the highest valid version wins; unresolved same-version conflicts are quarantined with their source references.
- **Same rules for both configurations:** only the widgets `schema` and `validator_run` change between the development extract and the full release.

In [ ]:
import re
from datetime import date, timedelta

# payment_day is the only business parameter. cutoff_ids lets a run publish C1, C2 or both:
# a C2-only run is what the recovery exercise (Q5) needs.
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "bdcc_g_dev")
dbutils.widgets.text("payment_day", "2026-01-31")
dbutils.widgets.text("cutoff_ids", "C1,C2")
dbutils.widgets.text("validator_run", "dev_initial")
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
payment_day = date.fromisoformat(dbutils.widgets.get("payment_day").strip())
cutoff_ids = [c.strip() for c in dbutils.widgets.get("cutoff_ids").split(",") if c.strip()]
validator_run = dbutils.widgets.get("validator_run").strip()
for value in (catalog, schema):
    assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value), "Use a simple catalog and schema identifier"
assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]*", validator_run), "Use letters, digits, dash or underscore"
assert cutoff_ids and set(cutoff_ids) <= {"C1", "C2"}, "cutoff_ids is C1, C2 or C1,C2"
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")
spark.sql("SET TIME ZONE 'UTC'")
validator_path = f"/Volumes/{catalog}/{schema}/landing/validator/{validator_run}"

# WHY derive both cutoffs from D: the brief defines them relative to D. Typing them separately
# could produce two cutoffs that do not belong to the same payment day.
# C1 = 08:00 UTC on D+1, C2 = 08:00 UTC on D+2. Only the payment day is a parameter.
CUTOFFS = {f"C{n}": f"{payment_day + timedelta(days=n)}T08:00:00Z" for n in (1, 2)}
# A tiny view with one row per selected cutoff. Joining a source to it evaluates the same rows
# once per cutoff, with one and the same statement.
selected = "\nUNION ALL\n".join(
    f"SELECT '{cutoff_id}' AS cutoff_id, '{CUTOFFS[cutoff_id]}' AS cutoff_utc" for cutoff_id in cutoff_ids)
spark.sql(f'''
CREATE OR REPLACE TEMP VIEW cutoffs AS
SELECT cutoff_id, cutoff_utc, try_cast(cutoff_utc AS TIMESTAMP) AS cutoff_ts
FROM (
{selected}
) c
''')
# Used below in REPLACE WHERE: only the cutoffs of this run are replaced.
cutoff_filter = "cutoff_id IN (" + ", ".join(f"'{c}'" for c in cutoff_ids) + ")"
print("Payment day D:", payment_day, "| cutoffs in this run:", {c: CUTOFFS[c] for c in cutoff_ids})

# The six sources the close needs (see 02_profile_sources for the two exclusions).
SOURCES = ["gateway_requests", "processor_events", "refund_events", "settlement_lines",
           "reference_mappings", "terminal_assignments"]

## 1. One policy for six sources

The statement below runs once per source. Read it top to bottom, it is Finance's close policy in order:

1. **available**: delivered at or before the cutoff. A delivery stamped exactly 08:00:00 is in, one stamped 08:00:01 is out.
2. **eligible**: usable, and effective by the cutoff.
3. **copies**: among eligible rows, identical content under one key and version is one fact. The earliest delivery is kept, the others are replays and point to the kept row.
4. **versions**: per key, the highest version available wins. Lower versions are superseded: a correction replaces a movement, it does not add one.
5. **conflict**: if the winning version still has two different contents, nobody can say which is right. Both rows go to quarantine with their source references and the fact has no value at this cutoff.

A conflict in a lower version that a clean higher version replaces is resolved by that version and is not quarantined.

In [ ]:
# How the statement maps to Finance's close policy (read the CTEs top to bottom):
#   available : delivered_ts <= cutoff_ts. "<=" because the dictionary says to include records
#               delivered exactly at the cutoff. A delivery at 08:00:01 fails the test.
#   staged    : REJECTED (failed a row check in notebook 01), NOT_YET_EFFECTIVE (delivered in
#               time, effective only after the cutoff), otherwise ELIGIBLE.
#   copies    : rows with the same key, version and content are numbered by delivery time.
#               copy_no = 1 is the kept row. The others are replays and remember the kept row
#               (kept_source_row_id), so they stay traceable. 'stage' is part of the partition,
#               so a rejected row can never be the kept copy of an eligible one.
#   versions  : per key, top_version = highest version among the kept eligible rows, and
#               contents_in_version = how many different contents that version has.
#   final CASE: the order gives every row exactly ONE status: not eligible -> its stage,
#               a copy -> REPLAY, a lower version -> SUPERSEDED, top version with two or more
#               contents -> QUARANTINED, otherwise CURRENT.
# WHY SUPERSEDED is tested before QUARANTINED: a conflict in an old version that a clean higher
# version replaced is resolved and must not block the close.
# WHY quarantine instead of picking one amount: nobody can say which is right. Choosing silently
# would hide a data-quality case from Finance.
# WHY one statement for six sources: the policy is the same for every fact, and stg_* gives all
# sources the same column names (business_key, version_no, payload_hash, effective_ts).
def snapshot_sql(name):
    return f'''
WITH available AS (
  SELECT c.cutoff_id, c.cutoff_ts, s.*
  FROM stg_{name} s
  JOIN cutoffs c ON s.delivered_ts <= c.cutoff_ts
),
staged AS (
  SELECT *,
         CASE WHEN validation_issue IS NOT NULL THEN 'REJECTED'
              WHEN effective_ts > cutoff_ts THEN 'NOT_YET_EFFECTIVE'
              ELSE 'ELIGIBLE' END AS stage
  FROM available
),
copies AS (
  SELECT *,
         row_number() OVER (PARTITION BY cutoff_id, stage, business_key, version_no, payload_hash
                            ORDER BY delivered_ts, source_row_id) AS copy_no,
         first_value(source_row_id) OVER (PARTITION BY cutoff_id, stage, business_key, version_no, payload_hash
                                          ORDER BY delivered_ts, source_row_id) AS kept_source_row_id
  FROM staged
),
versions AS (
  SELECT *,
         max(CASE WHEN stage = 'ELIGIBLE' AND copy_no = 1 THEN version_no END)
             OVER (PARTITION BY cutoff_id, business_key) AS top_version,
         count(CASE WHEN stage = 'ELIGIBLE' AND copy_no = 1 THEN 1 END)
             OVER (PARTITION BY cutoff_id, business_key, version_no) AS contents_in_version
  FROM copies
)
SELECT *,
       CASE WHEN stage <> 'ELIGIBLE' THEN stage
            WHEN copy_no > 1 THEN 'REPLAY'
            WHEN version_no < top_version THEN 'SUPERSEDED'
            WHEN contents_in_version > 1 THEN 'QUARANTINED'
            ELSE 'CURRENT' END AS row_status
FROM versions
'''

## 2. Publish the snapshots

One table per source, named `snap_` plus the source (for example `snap_processor_events`), with one partition per cutoff. Each cutoff is replaced as a whole in a single Delta commit:

- running this notebook twice gives the same rows, never duplicates
- running it for `cutoff_ids = C2` alone leaves the C1 snapshot untouched

**Why one table per source with one partition per cutoff.** The brief asks us to preserve both snapshots, and Q5 asks us to show that a rerun creates no duplicates and leaves C1 untouched. A partition per cutoff that is replaced as a whole gives both.

In [ ]:
# WHY a Delta table partitioned by cutoff_id: each cutoff is its own partition, so publishing C2
# never rewrites C1 ("preserve both snapshots").
# First statement : creates the empty table once. LIMIT 0 takes the columns from the query and
#                   no rows.
# Second statement: replaces exactly the cutoffs of this run in ONE atomic commit (REPLACE WHERE).
#                   A rerun gives the same rows, never duplicates. A failed run leaves the
#                   previous version in place.
# WHY not CREATE OR REPLACE TABLE: a C2-only run would drop the C1 snapshot.
# WHY not a plain INSERT: a rerun would double every row.
for name in SOURCES:
    query = snapshot_sql(name)
    spark.sql(f"CREATE TABLE IF NOT EXISTS snap_{name} USING DELTA PARTITIONED BY (cutoff_id) AS\n{query}\nLIMIT 0")
    spark.sql(f"INSERT INTO snap_{name} REPLACE WHERE {cutoff_filter}\n{query}")
    print(f"Published snap_{name} for {cutoff_ids}")

## 3. Is every row accounted for?

For each cutoff and source: rows delivered by the cutoff = rejected + not yet effective + replays + superseded + quarantined + current.

In [ ]:
# One audit row per cutoff and source: how many rows ended in each status.
# WHY: every delivered row must be accounted for (P1: raw = checked + rejected + duplicates).
# quarantined_rows counts rows, quarantined_keys counts facts (a conflict has at least two rows).
audit = "\nUNION ALL\n".join(f'''
SELECT cutoff_id, '{name}' AS source_table,
       count(*) AS delivered_by_cutoff,
       count_if(row_status = 'REJECTED') AS rejected,
       count_if(row_status = 'NOT_YET_EFFECTIVE') AS not_yet_effective,
       count_if(row_status = 'REPLAY') AS replays,
       count_if(row_status = 'SUPERSEDED') AS superseded_versions,
       count_if(row_status = 'QUARANTINED') AS quarantined_rows,
       count(DISTINCT CASE WHEN row_status = 'QUARANTINED' THEN business_key END) AS quarantined_keys,
       count_if(row_status = 'CURRENT') AS current_facts
FROM snap_{name}
GROUP BY cutoff_id''' for name in SOURCES)
spark.sql(f"CREATE OR REPLACE TABLE ingest_audit AS\n{audit}")

In [ ]:
%sql
-- The control totals of this run. Compare C1 and C2 line by line: a difference can only come
-- from deliveries after C1.
SELECT * FROM ingest_audit ORDER BY source_table, cutoff_id

In [ ]:
# The Docker validator computed the same control totals from the CSVs with different code.
expected = {(row["cutoff_id"], row["source_table"]): row for row in
            spark.read.option("header", True).option("inferSchema", False)
            .csv(f"{validator_path}/cutoff_profile.csv").collect()}
columns = ["delivered_by_cutoff", "rejected", "not_yet_effective", "replays", "superseded_versions",
           "quarantined_rows", "quarantined_keys", "current_facts"]
# Two checks per snapshot:
#   internal - the statuses add up to the rows delivered by the cutoff
#   external - every figure equals the validator's figure, computed independently in the container
checked = 0
for row in spark.sql("SELECT * FROM ingest_audit").collect():
    parts = ["rejected", "not_yet_effective", "replays", "superseded_versions", "quarantined_rows", "current_facts"]
    assert int(row["delivered_by_cutoff"]) == sum(int(row[p]) for p in parts), ("does not reconcile", row)
    control = expected[(row["cutoff_id"], row["source_table"])]
    for column in columns:
        assert int(row[column]) == int(control[column]), (row["cutoff_id"], row["source_table"], column,
                                                          int(row[column]), int(control[column]))
    checked += 1
print(f"{checked} snapshots reconcile row for row and agree with the validator's control totals.")

## 4. What the close reads

The close reads two kinds of views. The next cell creates both.

**The clean views.** There is one per source, named clean_ plus the source name, for example clean_processor_events. Each holds only the CURRENT facts, one row per key and cutoff. Timestamps are real timestamps and amounts are whole numbers of minor units, under the column names of the data dictionary. Every query on a clean view must filter on cutoff_id. Otherwise it mixes C1 and C2.

**The data-quality views.** dq_rejected, dq_replays and dq_quarantine hold the rows that did not make it into the close, each with its source_row_id and its delivery. dq_quarantine_cases lists one row per conflict. These views feed the exception queue.

**Why views here.** `clean_*` and `dq_*` only select from the stored `snap_*` tables, so they can never disagree with the published snapshot. The preserved data are the `snap_*` tables.

In [ ]:
# clean_<source>: what the close reads. Only CURRENT rows, under the dictionary's column names
# but with real types: the parsed column p_<column> replaces the raw text, version_no replaces
# version.
for name in SOURCES:
    raw_columns = spark.table(f"raw_{name}").columns
    snap_columns = spark.table(f"snap_{name}").columns
    select = []
    for column in raw_columns:
        if column == "version":
            select.append("version_no AS version")
        elif f"p_{column}" in snap_columns:
            select.append(f"p_{column} AS {column}")
        else:
            select.append(column)
    spark.sql(f'''
CREATE OR REPLACE VIEW clean_{name} AS
SELECT cutoff_id, cutoff_ts, business_key, {", ".join(select)}, delivered_ts
FROM snap_{name}
WHERE row_status = 'CURRENT'
''')

# The rows that did NOT enter the close, for all sources in one shape: cutoff, source, key,
# version, one reason-specific column and the source reference (source_row_id, delivery_id).
# These views feed the exception queue.
def union_of(status, extra):
    return "\nUNION ALL\n".join(
        f"SELECT cutoff_id, '{name}' AS source_table, business_key, version_no, {extra}, "
        f"source_row_id, delivery_id, delivered_ts FROM snap_{name} WHERE row_status = '{status}'"
        for name in SOURCES)

spark.sql("CREATE OR REPLACE VIEW dq_rejected AS\n" + union_of("REJECTED", "validation_issue AS reason"))
spark.sql("CREATE OR REPLACE VIEW dq_replays AS\n" + union_of("REPLAY", "kept_source_row_id"))
spark.sql("CREATE OR REPLACE VIEW dq_quarantine AS\n" + union_of("QUARANTINED", "contents_in_version"))
# One row per conflict (key and version) with all its source_row_ids: the source references the
# policy asks us to record for quarantined conflicts.
spark.sql('''
CREATE OR REPLACE VIEW dq_quarantine_cases AS
SELECT cutoff_id, source_table, business_key, version_no, count(*) AS n_rows,
       concat_ws(' ', sort_array(collect_list(source_row_id))) AS source_row_ids
FROM dq_quarantine
GROUP BY cutoff_id, source_table, business_key, version_no
''')

In [ ]:
# One row per key and cutoff, or the close would count a fact twice.
# WHY assert the grain: every later join assumes one row per key and cutoff. If that were false,
# the close would double a payment without any error message.
for name in SOURCES:
    repeated = spark.sql(f'''
SELECT count(*) AS n FROM (SELECT cutoff_id, business_key FROM clean_{name}
                           GROUP BY cutoff_id, business_key HAVING count(*) > 1) d''').first()["n"]
    assert repeated == 0, f"clean_{name} repeats a key"
print("Every clean view has one row per key and cutoff.")

## 5. What changed between the cutoffs?

The same statement, two cutoffs. Any difference comes from deliveries after C1, and each one can move a payment's status. This cell needs both cutoffs in the run.

In [ ]:
%sql
-- Every delivery after C1, grouped by source, delivery time and kind. 'new at C2' is the source
-- evidence for a payment whose status differs between the cutoffs. Rows delivered after C2
-- (08:00:01) are in neither close.
WITH delivered AS (
  SELECT 'processor_events' AS source_table, delivered_at, delivered_ts, event_type AS kind, version FROM stg_processor_events
  UNION ALL
  SELECT 'refund_events', delivered_at, delivered_ts, refund_status, version FROM stg_refund_events
  UNION ALL
  SELECT 'settlement_lines', delivered_at, delivered_ts, posting_type, version FROM stg_settlement_lines
  UNION ALL
  SELECT 'reference_mappings', delivered_at, delivered_ts, from_system, version FROM stg_reference_mappings
)
SELECT d.source_table, d.delivered_at,
       CASE WHEN d.delivered_ts <= c2.cutoff_ts THEN 'new at C2' ELSE 'after C2, in neither close' END AS seen,
       d.kind, d.version, count(*) AS n_rows
FROM delivered d
JOIN cutoffs c1 ON c1.cutoff_id = 'C1'
JOIN cutoffs c2 ON c2.cutoff_id = 'C2'
WHERE d.delivered_ts > c1.cutoff_ts
GROUP BY d.source_table, d.delivered_at, CASE WHEN d.delivered_ts <= c2.cutoff_ts THEN 'new at C2' ELSE 'after C2, in neither close' END,
         d.kind, d.version
ORDER BY d.source_table, d.delivered_at, d.kind, d.version

In [ ]:
%sql
-- Two corrected captures traced through both snapshots: at C1 version 1 is CURRENT, at C2
-- version 2 is CURRENT and version 1 is SUPERSEDED.
SELECT cutoff_id, business_key, version_no, row_status, source_row_id, delivery_id, delivered_at,
       event_type, amount_minor, currency_cd
FROM snap_processor_events
WHERE business_key IN (SELECT business_key FROM snap_processor_events
                       WHERE row_status = 'SUPERSEDED' ORDER BY business_key LIMIT 2)
ORDER BY business_key, cutoff_id, version_no

In [ ]:
%sql
-- The open conflicts, one line per case, with the source rows to hand to Operations.
SELECT * FROM dq_quarantine_cases ORDER BY cutoff_id, source_table, business_key LIMIT 10

## 6. Rerun and look again

Run section 2 a second time, then the next cell. The row counts per cutoff do not move, and the table history shows one commit per publication.

In [ ]:
%sql
-- Rows per cutoff. Run section 2 again and repeat this cell: the numbers must not move.
SELECT cutoff_id, count(*) AS n_rows, count_if(row_status = 'CURRENT') AS current_facts
FROM snap_processor_events
GROUP BY cutoff_id
ORDER BY cutoff_id

In [ ]:
%sql
-- The Delta history of the table: each publication is one commit.
DESCRIBE HISTORY snap_processor_events

## Key findings to carry forward

- A snapshot is a decision about every delivered row, not a filter. Each row has one status, and the statuses add up to the delivery.
- Corrections replace, they do not add: a version 2 capture supersedes version 1.
- A same-version conflict has no value until a higher version resolves it. Quarantined facts and rejected rows must reach the exception queue: a payment whose capture is unusable cannot be signed off as if it had none.
- C1 stays as published. C2 is a second partition, not an update.

**Assumptions that could change Finance's answer**

1. If the highest available version of a fact is in conflict, the fact is quarantined. We do not fall back to an older version that the conflict was meant to replace.
2. A row whose amount or effective time cannot be read is rejected, not repaired.
3. A gateway request counts as effective at its request time.

**Next:** reconstruct each payment at C1 and C2 from the `clean_*` views.